# 使用 PPE 数据集在 CPU 模式下训练 DEIMv2 模型

本教程介绍如何将 YOLO 格式的 PPE 数据集转换为 COCO 格式，并用于 DEIMv2 模型训练（CPU模式）。

**数据集信息:**
- 来源: Roboflow PPE Dataset
- 类别: Helmet, NoHelmet, NoVest, Vest
- 训练集: 1829 张图像
- 验证集: 457 张图像

**注意**: CPU训练速度较慢，建议使用较小的模型（HGNetv2-Pico/Femto/Atto）和较少的epoch进行训练。

## 1. YOLO 转 COCO 格式

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime
from PIL import Image
from tqdm import tqdm
import yaml

def yolo_to_coco(yolo_data_dir, output_dir):
    yolo_path = Path(yolo_data_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    yaml_path = yolo_path / 'data.yaml'
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data_yaml = yaml.safe_load(f)
    class_names = data_yaml.get('names', [])
    print(f'类别: {class_names}')

    categories = [{'id': i, 'name': name, 'supercategory': name} for i, name in enumerate(class_names)]

    for split in ['train', 'valid']:
        images_dir = yolo_path / split / 'images'
        labels_dir = yolo_path / split / 'labels'
        if not images_dir.exists():
            continue

        print(f'处理 {split} 集...')
        coco_output = {'info': {'description': 'Converted from YOLO', 'version': '1.0', 'year': datetime.now().year},
                       'categories': categories, 'images': [], 'annotations': []}

        image_id = 0
        annotation_id = 0
        image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')) + list(images_dir.glob('*.jpeg'))

        for img_file in tqdm(image_files, desc=f'Converting {split}'):
            with Image.open(img_file) as img:
                width, height = img.size

            coco_output['images'].append({'id': image_id, 'file_name': img_file.name,
                                           'width': width, 'height': height})

            label_file = labels_dir / (img_file.stem + '.txt')
            if label_file.exists():
                with open(label_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5:
                            continue
                        class_id = int(parts[0])
                        x_c, y_c, bw, bh = map(float, parts[1:5])
                        x_min = (x_c - bw/2) * width
                        y_min = (y_c - bh/2) * height
                        bw *= width
                        bh *= height
                        coco_output['annotations'].append({'id': annotation_id, 'image_id': image_id,
                            'category_id': class_id, 'bbox': [x_min, y_min, bw, bh],
                            'area': bw * bh, 'iscrowd': 0})
                        annotation_id += 1
            image_id += 1

        output_json = output_path / f'{split}.json'
        with open(output_json, 'w') as f:
            json.dump(coco_output, f)
        print(f'  图像: {len(coco_output["images"])}, 标注: {len(coco_output["annotations"])}')
        print(f'  保存到: {output_json}')

    return output_path

In [ ]:
# 设置数据集路径
yolo_data_dir = r'D:\AI\Datasets\Dataset of Personal Protective Equipment (PPE)\20250731-ppe2286y'
output_dir = yolo_data_dir

# 检查是否已存在COCO格式的json文件
train_json = Path(output_dir) / 'train.json'
valid_json = Path(output_dir) / 'valid.json'

if train_json.exists() and valid_json.exists():
    print(f'COCO格式文件已存在，跳过转换:')
    print(f'  - {train_json}')
    print(f'  - {valid_json}')
else:
    yolo_to_coco(yolo_data_dir, output_dir)
    print('格式转换完成!')

## 2. 创建 CPU 训练配置文件

由于CPU训练资源有限，我们需要创建一个轻量级的配置文件。这里使用HGNetv2-Atto模型（最小模型，0.5M参数）。

In [ ]:
import yaml

# 创建 PPE 数据集配置文件 (CPU版本) - 修正版
ppe_config = {
    '__include__': [
        '../dataset/coco_detection.yml',
        '../runtime.yml',
        '../base/dataloader.yml',
        '../base/optimizer.yml',
        '../base/deimv2.yml'
    ],
    'output_dir': './outputs/deimv2_hgnetv2_atto_ppe_cpu',
    'num_classes': 4,
    'remap_mscoco_category': False,
    
    # 使用HGNetv2-Atto模型
    'HGNetv2': {
        'name': 'Atto',
        'return_idx': [2],  # 只使用stage3输出
        'freeze_at': -1,
        'freeze_norm': False,
        'use_lab': True
    },
    
    # 使用LiteEncoder (Atto模型专用)
    'DEIM': {
        'encoder': 'LiteEncoder'
    },
    
    'LiteEncoder': {
        'in_channels': [256],
        'feat_strides': [16],
        'hidden_dim': 64,
        'expansion': 0.34,
        'depth_mult': 0.5,
        'act': 'silu'
    },
    
    'DEIMTransformer': {
        'feat_channels': [64, 64],
        'feat_strides': [16, 32],
        'hidden_dim': 64,
        'num_levels': 2,
        'num_points': [4, 2],
        'num_layers': 3,
        'eval_idx': -1,
        'num_queries': 100,
        'dim_feedforward': 160,
        'share_bbox_head': True,
        'use_gateway': False
    },
    
    # 优化器配置
    'optimizer': {
        'type': 'AdamW',
        'params': [
            {'params': '^(?=.*backbone)(?!.*norm|bn).*$', 'lr': 0.0001},
            {'params': '^(?=.*backbone)(?=.*norm|bn).*$', 'lr': 0.0001, 'weight_decay': 0.0},
            {'params': '^(?=.*(?:encoder|decoder))(?=.*(?:norm|bn)).*$', 'weight_decay': 0.0}
        ],
        'lr': 0.0004,
        'betas': [0.9, 0.999],
        'weight_decay': 0.0001
    },
    
    # 训练参数
    'epoches': 50,
    'flat_epoch': 27,
    'no_aug_epoch': 8,
    'eval_spatial_size': [320, 320]
}

# 训练数据增强 (与官方Atto配置一致)
ppe_config['train_dataloader'] = {
    'type': 'DataLoader',
    'dataset': {
        'type': 'CocoDetection',
        'img_folder': r'D:\AI\Datasets\Dataset of Personal Protective Equipment (PPE)\20250731-ppe2286y\train\images',
        'ann_file': r'D:\AI\Datasets\Dataset of Personal Protective Equipment (PPE)\20250731-ppe2286y\train.json',
        'return_masks': False,
        'transforms': {
            'type': 'Compose',
            'ops': [
                {'type': 'Mosaic', 'output_size': 160, 'rotation_range': 10, 'translation_range': [0.1, 0.1], 
                 'scaling_range': [0.5, 1.5], 'probability': 1.0, 'fill_value': 0, 
                 'use_cache': True, 'max_cached_images': 20, 'random_pop': True},
                {'type': 'RandomPhotometricDistort', 'p': 0.5},
                {'type': 'RandomZoomOut', 'fill': 0},
                {'type': 'RandomIoUCrop', 'p': 0.8},
                {'type': 'SanitizeBoundingBoxes', 'min_size': 12},
                {'type': 'RandomHorizontalFlip'},
                {'type': 'Resize', 'size': [320, 320]},
                {'type': 'SanitizeBoundingBoxes', 'min_size': 12},
                {'type': 'ConvertPILImage', 'dtype': 'float32', 'scale': True},
                {'type': 'ConvertBoxes', 'fmt': 'cxcywh', 'normalize': True}
            ],
            'policy': {
                'epoch': [4, 27, 42],
                'mosaic_prob': 0.3
            }
        }
    },
    'shuffle': True,
    'num_workers': 2,
    'drop_last': True,
    'total_batch_size': 4,
    'collate_fn': {
        'type': 'BatchImageCollateFunction',
        'mixup_prob': 0.0,
        'mixup_epochs': [40000, 15000],
        'copyblend_prob': 0.0,
        'copyblend_epochs': [40000, 15000],
        'stop_epoch': 42,
        'ema_restart_decay': 0.9999,
        'base_size': 320,
        'base_size_repeat': None
    }
}

# 验证数据加载器
ppe_config['val_dataloader'] = {
    'type': 'DataLoader',
    'dataset': {
        'type': 'CocoDetection',
        'img_folder': r'D:\AI\Datasets\Dataset of Personal Protective Equipment (PPE)\20250731-ppe2286y\valid\images',
        'ann_file': r'D:\AI\Datasets\Dataset of Personal Protective Equipment (PPE)\20250731-ppe2286y\valid.json',
        'return_masks': False,
        'transforms': {
            'type': 'Compose',
            'ops': [
                {'type': 'Resize', 'size': [320, 320]},
                {'type': 'ConvertPILImage', 'dtype': 'float32', 'scale': True}
            ]
        }
    },
    'shuffle': False,
    'num_workers': 2,
    'drop_last': False,
    'total_batch_size': 4,
    'collate_fn': {'type': 'BatchImageCollateFunction'}
}

# Matcher配置
ppe_config['DEIMCriterion'] = {
    'losses': ['mal', 'boxes'],
    'matcher': {
        'change_matcher': True,
        'iou_order_alpha': 4.0,
        'matcher_change_epoch': 38
    }
}

# 保存配置文件
config_path = '../configs/deimv2/deimv2_hgnetv2_atto_ppe_cpu.yml'
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(ppe_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f'配置文件已保存到: {config_path}')
print(f'训练参数:')
print(f'  - 模型: HGNetv2-Atto (0.5M参数)')
print(f'  - Encoder: LiteEncoder')
print(f'  - 类别数: 4 (Helmet, NoHelmet, NoVest, Vest)')
print(f'  - Epoch: 50')
print(f'  - Batch size: 4')
print(f'  - 输入尺寸: 320x320')
print(f'  - 设备: CPU')

In [ ]:
import yaml

# 创建头盔检测数据集配置文件 (CPU版本)
helmet_config = {
    '__include__': [
        '../dataset/coco_detection.yml',
        '../runtime.yml',
        '../base/dataloader.yml',
        '../base/optimizer.yml',
        '../base/deimv2.yml'
    ],
    'output_dir': './outputs/deimv2_hgnetv2_atto_helmet_cpu',
    'num_classes': 3,  # 3个类别: supercategory, Helmet, NOHelmet
    'remap_mscoco_category': False,
    
    # 使用HGNetv2-Atto模型
    'HGNetv2': {
        'name': 'Atto',
        'return_idx': [2],
        'freeze_at': -1,
        'freeze_norm': False,
        'use_lab': True
    },
    
    # 使用LiteEncoder
    'DEIM': {
        'encoder': 'LiteEncoder'
    },
    
    'LiteEncoder': {
        'in_channels': [256],
        'feat_strides': [16],
        'hidden_dim': 64,
        'expansion': 0.34,
        'depth_mult': 0.5,
        'act': 'silu'
    },
    
    'DEIMTransformer': {
        'feat_channels': [64, 64],
        'feat_strides': [16, 32],
        'hidden_dim': 64,
        'num_levels': 2,
        'num_points': [4, 2],
        'num_layers': 3,
        'eval_idx': -1,
        'num_queries': 100,
        'dim_feedforward': 160,
        'share_bbox_head': True,
        'use_gateway': False
    },
    
    # 优化器配置
    'optimizer': {
        'type': 'AdamW',
        'params': [
            {'params': '^(?=.*backbone)(?!.*norm|bn).*$', 'lr': 0.0001},
            {'params': '^(?=.*backbone)(?=.*norm|bn).*$', 'lr': 0.0001, 'weight_decay': 0.0},
            {'params': '^(?=.*(?:encoder|decoder))(?=.*(?:norm|bn)).*$', 'weight_decay': 0.0}
        ],
        'lr': 0.0004,
        'betas': [0.9, 0.999],
        'weight_decay': 0.0001
    },
    
    # 训练参数 (图片尺寸640x640)
    'epoches': 50,
    'flat_epoch': 27,
    'no_aug_epoch': 8,
    'eval_spatial_size': [640, 640]
}

# 训练数据加载器
helmet_config['train_dataloader'] = {
    'type': 'DataLoader',
    'dataset': {
        'type': 'CocoDetection',
        'img_folder': r'D:\AI\Datasets\helmet detection dataset.v10i.coco\train',
        'ann_file': r'D:\AI\Datasets\helmet detection dataset.v10i.coco\train\_annotations.coco.json',
        'return_masks': False,
        'transforms': {
            'type': 'Compose',
            'ops': [
                {'type': 'Mosaic', 'output_size': 320, 'rotation_range': 10, 'translation_range': [0.1, 0.1], 
                 'scaling_range': [0.5, 1.5], 'probability': 1.0, 'fill_value': 0, 
                 'use_cache': True, 'max_cached_images': 20, 'random_pop': True},
                {'type': 'RandomPhotometricDistort', 'p': 0.5},
                {'type': 'RandomZoomOut', 'fill': 0},
                {'type': 'RandomIoUCrop', 'p': 0.8},
                {'type': 'SanitizeBoundingBoxes', 'min_size': 12},
                {'type': 'RandomHorizontalFlip'},
                {'type': 'Resize', 'size': [640, 640]},
                {'type': 'SanitizeBoundingBoxes', 'min_size': 12},
                {'type': 'ConvertPILImage', 'dtype': 'float32', 'scale': True},
                {'type': 'ConvertBoxes', 'fmt': 'cxcywh', 'normalize': True}
            ],
            'policy': {
                'epoch': [4, 27, 42],
                'mosaic_prob': 0.3
            }
        }
    },
    'shuffle': True,
    'num_workers': 2,
    'drop_last': True,
    'total_batch_size': 4,
    'collate_fn': {
        'type': 'BatchImageCollateFunction',
        'mixup_prob': 0.0,
        'mixup_epochs': [40000, 15000],
        'copyblend_prob': 0.0,
        'copyblend_epochs': [40000, 15000],
        'stop_epoch': 42,
        'ema_restart_decay': 0.9999,
        'base_size': 640,
        'base_size_repeat': None
    }
}

# 验证数据加载器
helmet_config['val_dataloader'] = {
    'type': 'DataLoader',
    'dataset': {
        'type': 'CocoDetection',
        'img_folder': r'D:\AI\Datasets\helmet detection dataset.v10i.coco\valid',
        'ann_file': r'D:\AI\Datasets\helmet detection dataset.v10i.coco\valid\_annotations.coco.json',
        'return_masks': False,
        'transforms': {
            'type': 'Compose',
            'ops': [
                {'type': 'Resize', 'size': [640, 640]},
                {'type': 'ConvertPILImage', 'dtype': 'float32', 'scale': True}
            ]
        }
    },
    'shuffle': False,
    'num_workers': 2,
    'drop_last': False,
    'total_batch_size': 4,
    'collate_fn': {'type': 'BatchImageCollateFunction'}
}

# Matcher配置
helmet_config['DEIMCriterion'] = {
    'losses': ['mal', 'boxes'],
    'matcher': {
        'change_matcher': True,
        'iou_order_alpha': 4.0,
        'matcher_change_epoch': 38
    }
}

# 保存配置文件
config_path = '../configs/deimv2/deimv2_hgnetv2_atto_helmet_cpu.yml'
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(helmet_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f'✅ 头盔检测配置文件已保存到: {config_path}')
print(f'\n📊 数据集信息:')
print(f'  - 类别数: 3 (Helmet, NOHelmet)')
print(f'  - 图片尺寸: 640x640')
print(f'  - 训练/验证: 自动从COCO标注中读取')
print(f'\n🚀 训练命令:')
print(f'  python train.py -c configs/deimv2/deimv2_hgnetv2_atto_helmet_cpu.yml -d cpu --seed 0')

## 3. 开始 CPU 训练

在CPU模式下训练需要使用单进程模式，不需要`torchrun`命令。注意：CPU训练速度较慢！

In [ ]:
import subprocess
import os

# 设置环境变量以强制使用CPU
os.environ['CUDA_VISIBLE_DEVICES'] = ''

# 训练命令（CPU模式）
# 注意：不使用 --use-amp（混合精度在CPU上效率低）
# 不使用 torchrun，直接使用 python 运行单进程
train_cmd = [
    'python', '../train.py',
    '-c', '../configs/deimv2/deimv2_hgnetv2_atto_ppe_cpu.yml',
    '-d', 'cpu',  # 强制使用CPU
    '--seed', '0'
]

print('开始训练...')
print(' '.join(train_cmd))
print('\n注意：')
print('1. CPU训练速度较慢，请耐心等待')
print('2. 可以按 Ctrl+C 中断训练')
print('3. 训练日志将保存到 outputs/deimv2_hgnetv2_atto_ppe_cpu/')

# 运行训练（如需后台运行可注释此行）
# subprocess.run(train_cmd, cwd='..')

## 4. 测试训练好的模型

在CPU模式下加载训练好的模型并进行推理。

In [ ]:
import torch
import os
from PIL import Image, ImageDraw
from torchvision import transforms
import matplotlib.pyplot as plt

# 类别定义
PPE_CLASSES = {0: 'Helmet', 1: 'NoHelmet', 2: 'NoVest', 3: 'Vest'}
COLORS = {0: '#00FF00', 1: '#FF0000', 2: '#FFA500', 3: '#00BFFF'}

# 设置设备为CPU
device = torch.device('cpu')
print(f'使用设备: {device}')

# 检查是否有训练好的权重
checkpoint_path = '../outputs/deimv2_hgnetv2_atto_ppe_cpu/best.pth'
if not os.path.exists(checkpoint_path):
    checkpoint_path = '../outputs/deimv2_hgnetv2_atto_ppe_cpu/last.pth'

print(f'权重文件路径: {checkpoint_path}')
print(f'权重文件存在: {os.path.exists(checkpoint_path)}')

In [ ]:
# 使用配置文件进行推理
# 注意：这里假设您已经训练好了模型

import sys
sys.path.insert(0, '..')

from engine.core import YAMLConfig
from engine.solver import TASKS
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch

def test_ppe_detection(image_path, config_path, checkpoint_path, conf_threshold=0.5):
    """测试PPE检测"""
    
    # 加载配置和模型
    cfg = YAMLConfig(config_path)
    
    # 设置设备为CPU
    cfg.device = 'cpu'
    
    # 创建solver
    solver = TASKS[cfg.yaml_cfg['task']](cfg)
    
    # 加载权重
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        solver.model.load_state_dict(checkpoint.get('model', checkpoint), strict=False)
        print(f'已加载权重: {checkpoint_path}')
    else:
        print(f'权重文件不存在: {checkpoint_path}')
        print('请先完成训练或下载预训练权重')
        return None
    
    solver.model.eval()
    
    # 加载图像
    image = Image.open(image_path).convert('RGB')
    orig_w, orig_h = image.size
    
    # 预处理
    transform = transforms.Compose([
        transforms.Resize((640, 640)),
        transforms.ToTensor()
    ])
    input_tensor = transform(image).unsqueeze(0)
    
    # 推理
    with torch.no_grad():
        outputs = solver.model(input_tensor, orig_target_sizes=torch.tensor([[orig_h, orig_w]]))
    
    # 绘制结果
    draw = ImageDraw.Draw(image)
    for label, bbox, score in zip(outputs[0]['labels'], outputs[0]['boxes'], outputs[0]['scores']):
        if score.item() >= conf_threshold:
            class_id = label.item()
            x1, y1, x2, y2 = bbox.tolist()
            color = COLORS.get(class_id, '#FFFFFF')
            name = PPE_CLASSES.get(class_id, f'Class {class_id}')
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text((x1, y1 - 15), f'{name} {score:.2f}', fill=color)
    
    return image

# 示例测试（请先完成训练）
test_img = r'D:\AI\Datasets\Dataset of Personal Protective Equipment (PPE)\20250731-ppe2286y\valid\images\4_jpg.rf.89531bad47d2500e5e8229a6b4a8f50e.jpg'
config_path = '../configs/deimv2/deimv2_hgnetv2_atto_ppe_cpu.yml'
print(os.path.exists(checkpoint_path))
print(os.path.exists(test_img))
if os.path.exists(checkpoint_path) and os.path.exists(test_img):
    result = test_ppe_detection(test_img, config_path, checkpoint_path, 0.4)
    if result:
        plt.figure(figsize=(12, 8))
        plt.imshow(result)
        plt.axis('off')
        plt.show()
else:
    print('请确保权重文件和测试图像存在后再运行测试')

请确保权重文件和测试图像存在后再运行测试
